# DBT Wrapper - Utils

Shared utilities loaded by all wrappers via `%run ./dbt_wrapper_utils` after `restartPython()`.

## Sections

| Section | Functions |
|---|---|
| **Monitoring / Metrics** | `create_successful_metrics`, `create_failure_metrics`, `run_dbt_stage`, `send_stage_log`, `run_dbt_test` |
| **DBT Setup** | `setup_dbt_env`, `build_dbt_commands`, `add_date_flag`, `install_dbt_deps` |
| **Monitoring Bridge** | `build_extra_info`, `build_scala_tags`, `build_common_vars`, `build_base_execution_arguments` |
| **Widgets** | `setup_common_widgets` |
| **Checkpoint** | `get_last_successful_date`, `generate_date_range`, `update_checkpoint_file` |
| **Table Init** | `run_table_init` |


### Setup

Imports, JVM bridge, and Scala MonitoringExporter initialisation.
These cells run automatically when the notebook is loaded via `%run`.


In [0]:
import json
import logging
import os
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from dbt.cli.main import dbtRunner, dbtRunnerResult
import tempfile

In [0]:
jvm = spark._jvm
HashMap = jvm.scala.collection.mutable.HashMap
LocalDateTime = jvm.java.time.LocalDateTime
ZoneOffset = jvm.java.time.ZoneOffset
Log = jvm.com.brisa.monitoring.model.Log
LogStatusType = jvm.com.brisa.monitoring.model.LogStatusType
MonitoringExporter = jvm.com.brisa.monitoring.MonitoringExporter

### Monitoring / Metrics

Helper functions that build Scala HashMaps with model-level execution metrics,
and `run_dbt_stage` which wraps every dbt invocation with RUNNING/SUCCESSFUL/FAILED monitoring logs.


In [0]:
def create_successful_models_metrics(runner_result: dbtRunnerResult):
    """Return a Scala HashMap of {model_name: {elapsed: seconds}} for every successful model."""
    metrics = HashMap()
    for model_result in runner_result.result.results:
        if model_result.status.value == "success":
            time = HashMap()
            time.put("elapsed", model_result.execution_time)
            metrics.put(model_result.node.name, time)
    return metrics


def create_successful_metrics(runner_result: dbtRunnerResult):
    """Return monitoring metrics for a successful dbt run."""
    metrics = HashMap()
    metrics.put("successful_models", create_successful_models_metrics(runner_result))
    return metrics


def create_failure_metrics(runner_result: dbtRunnerResult):
    """Return monitoring metrics for a failed dbt run (failed model names and truncated error messages)."""
    metrics = HashMap()
    if runner_result.result:
        metrics.put("successful_models", create_successful_models_metrics(runner_result))
        failed_models = [
            r.node.name for r in runner_result.result.results
            if r.status.value != "success"
        ]
        metrics.put("failed_models", failed_models)
        errors = HashMap()
        for r in runner_result.result.results:
            if r.status.value != "success" and r.message:
                errors.put(r.node.name, r.message[:200])
        metrics.put("errors", errors)
    elif runner_result.exception:
        metrics.put("errors", str(runner_result.exception)[:1000])
    else:
        metrics.put("errors", "Unknown errors")
    return metrics


def send_stage_log(stage, status, message, metrics=None):
    """Send a single monitoring log entry with the current execution context.

    Args:
        stage:   Stage identifier used in the monitoring log.
        status:  LogStatusType value (RUNNING, SUCCESSFUL, FAILED).
        message: Human-readable message for the log entry.
        metrics: Optional Scala HashMap of metrics. Defaults to an empty HashMap.
    """
    MonitoringExporter.sendLog(Log(
        tool, process, job_id, run_id, execution_arguments, scala_tags,
        stage, status, metrics if metrics is not None else HashMap(), extra_info,
        message, LocalDateTime.now(ZoneOffset.UTC)
    ))


In [0]:
def run_dbt_stage(commands, stage, start_msg, success_msg, fail_msg, use_metrics=True):
    """Invoke a dbt command wrapped with RUNNING/SUCCESSFUL/FAILED monitoring logs.

    Args:
        commands:     dbt CLI args, e.g. ["run", "--target", target, "--select", path].
        stage:        Stage identifier used in the monitoring log.
        start_msg:    Message sent with the RUNNING log.
        success_msg:  Message sent on success.
        fail_msg:     Message sent on failure (also on exceptions).
        use_metrics:  Attach model-level metrics to the log (default True).
    Raises:
        Exception: re-raises any error after sending a FAILED log.
    """
    running_log = Log(
        tool, process, job_id, run_id, execution_arguments, scala_tags,
        stage, LogStatusType.RUNNING, HashMap(), extra_info,
        start_msg, LocalDateTime.now(ZoneOffset.UTC)
    )
    logging.info("[dbt:%s] %s", stage, start_msg)
    MonitoringExporter.sendLog(running_log)

    try:
        logging.info("[dbt:%s] Command: dbt %s", stage, " ".join(str(c) for c in commands))
        result = dbt_runner.invoke(commands)

        if result.success:
            status = LogStatusType.SUCCESSFUL
            message = success_msg
            metrics = create_successful_metrics(result) if use_metrics else HashMap()
        else:
            status = LogStatusType.FAILED
            message = fail_msg
            metrics = create_failure_metrics(result) if use_metrics else HashMap()

        logging.info("[dbt:%s] %s: %s", stage, "OK" if result.success else "FAILED", message)
        log = Log(
            tool, process, job_id, run_id, execution_arguments, scala_tags,
            stage, status, metrics, extra_info,
            message, LocalDateTime.now(ZoneOffset.UTC)
        )
        MonitoringExporter.sendLog(log)

    except Exception as e:
        error_metrics = HashMap()
        logging.error("[dbt:%s] ERROR: %s", stage, e)
        error_metrics.put("errors", str(e)[:1000])
        log = Log(
            tool, process, job_id, run_id, execution_arguments, scala_tags,
            stage, LogStatusType.FAILED, error_metrics, extra_info,
            fail_msg, LocalDateTime.now(ZoneOffset.UTC)
        )
        MonitoringExporter.sendLog(log)
        raise

    if status == LogStatusType.FAILED:
        error_msgs = []
        if result.result:
            error_msgs = [
                r.message for r in result.result.results
                if r.status.value != "success" and r.message
            ]
        details = " | ".join(error_msgs) if error_msgs else fail_msg
        raise Exception(f"Failed during stage '{stage}': {details}")

    return result

In [0]:
def run_dbt_test(dbt_test):
    """Run dbt test if dbt_test is True. A test failure emits a warning without failing the job."""
    if not dbt_test:
        logging.info("dbt_test parameter was not set to true")
        return
    commands = build_dbt_commands(target, "test", flags=extra_flags.get("dbt_test", {}))
    try:
        run_dbt_stage(
            commands, "test",
            "Running dbt tests", "Tests passed", "Tests failed",
            use_metrics=False
        )
    except Exception as e:
        logging.warning("dbt test did not pass — job will continue: %s", e)

### DBT Setup

Functions that configure the dbt environment and build dbt CLI commands.


In [0]:
def setup_dbt_env():
    """Set DBT_PROJECT_DIR, DBT_PROFILES_DIR and DBT_LOG_PATH from widgets. Sets up DBT_PACKAGES_INSTALL_PATH. Return the run timestamp."""
    timestamp = datetime.now(ZoneInfo("Europe/Lisbon")).strftime("%Y%m%d_%H%M%S")
    os.environ["DBT_PROJECT_DIR"] = dbutils.widgets.get("dbt_project")
    os.environ["DBT_PROFILES_DIR"] = dbutils.widgets.get("dbt_project")
    os.environ["DBT_LOG_PATH"] = (
        f"{dbutils.widgets.get('dbt_project')}/logs/"
        f"{timestamp}_{dbutils.widgets.get('run_id')}"
    )
    
    # Local DBT Packages install directory.
    packages_dir = tempfile.mkdtemp(prefix=f"{timestamp}_dbt_packages")
    os.environ["DBT_PACKAGES_INSTALL_PATH"] = packages_dir

    # Stop metadata cache as it leads to many unnecessary queries
    os.environ["DBT_POPULATE_CACHE"] = "False"

    return timestamp


In [0]:
def build_dbt_commands(target, command="run", model_path=None, flags=None):
    """Build the dbt CLI command list.

    Args:
        target:     dbt profile target (e.g. "prod").
        command:    dbt sub-command: "run", "test", "deps", etc.
        model_path: Optional --select path.
        flags:      Optional dict of extra flags; dict/list values are JSON-serialised.
    """
    commands = [command, "--target", target]
    if model_path:
        commands += ["--select", model_path]
    if flags:
        for key, value in flags.items():
            commands.append(f"--{key}")
            if isinstance(value, (dict, list)):
                commands.append(json.dumps(value))
            elif value != "":
                commands.append(str(value))
    return commands


In [0]:
def add_date_flag(model_path, extra_flags, date):
    """Inject exec_date into extra_flags[model_path]["vars"], creating nested keys as needed."""
    if model_path in extra_flags:
        if "vars" in extra_flags[model_path]:
            extra_flags[model_path]["vars"]["exec_date"] = date
        else:
            extra_flags[model_path]["vars"] = {"exec_date": date}
    else:
        extra_flags[model_path] = {"vars": {"exec_date": date}}


def install_dbt_deps():
    """Install dbt packages only when packages.yml or packages.yaml exists in the dbt project.

    Projects that do not use any dbt package skip this step automatically.
    Projects using the SCD package (or any other) must declare it in packages.yml.
    """
    dbt_project_path = os.environ.get("DBT_PROJECT_DIR", "")
    has_packages = (
        os.path.exists(os.path.join(dbt_project_path, "packages.yml")) or
        os.path.exists(os.path.join(dbt_project_path, "packages.yaml"))
    )
    if not has_packages:
        print("[INFO] No packages.yml found in dbt project. Skipping dbt deps.")
        return
    run_dbt_stage(
        commands=["deps"],
        stage="package",
        start_msg="Installing dbt packages",
        success_msg="dbt packages installed successfully",
        fail_msg="Failed to install dbt packages",
        use_metrics=False
    )


### Monitoring Bridge

Functions that read Databricks context and widget values to build the
shared monitoring log arguments used by every `Log` and `run_dbt_stage` call.


In [0]:
def build_extra_info():
    """Build a Scala HashMap with the Databricks run URL and run ID for monitoring log extra_info."""
    extra_info = HashMap()
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
        org_id = ctx.tags().get("orgId").get()
        db_job_id = ctx.tags().get("jobId").get()
        db_run_id = ctx.tags().get("jobRunId").get()
        extra_info.put(
            "run_url",
            f"https://{workspace_url}/?o={org_id}#job/{db_job_id}/run/{db_run_id}"
        )
        extra_info.put("run_id", db_run_id)
    except Exception as e:
        logging.warning("Could not build run_url for extra_info: %s", e)
    return extra_info

In [0]:
def build_scala_tags(tags):
    """Convert a Python dict of tags into a Scala HashMap for monitoring logs."""
    scala_tags = HashMap()
    for key, value in tags.items():
        scala_tags.put(key, value)
    return scala_tags


In [0]:
def build_common_vars():
    """Read shared widget values and return (tool, process, job_id, run_id, target, dbt_runner)."""
    tool = "DBT"
    process = dbutils.widgets.get("job_name")
    job_id = dbutils.widgets.get("job_id")
    run_id = dbutils.widgets.get("run_id")
    target = dbutils.widgets.get("target")
    dbt_runner = dbtRunner()
    return tool, process, job_id, run_id, target, dbt_runner


In [0]:
def build_base_execution_arguments(model_paths, target, extra_flags):
    """Build the Scala HashMap of execution arguments common to all wrappers.

    Includes: properties_file, model_paths, target, dbt_project, extra_flags.
    """
    execution_arguments = HashMap()
    execution_arguments.put(
        "properties_file", dbutils.widgets.get("properties_file")
    )
    execution_arguments.put("model_paths", model_paths)
    execution_arguments.put("target", target)
    execution_arguments.put(
        "dbt_project", dbutils.widgets.get("dbt_project")
    )
    execution_arguments.put("extra_flags", extra_flags)
    return execution_arguments


### Widgets

Declares and parses the widgets shared by all wrappers.


In [0]:
def setup_common_widgets():
    """Declare and parse the widgets shared by all wrappers.

    Widgets declared here:
        dbt_project     - path to the DBT project.
        model_paths     - JSON array of model paths to run in order (default: ["silver","gold"]).
        extra_flags     - JSON object of extra dbt flags keyed by model_path (default: {}).
        tags            - JSON object of monitoring tags (default: {}).
        properties_file - path to the Monitoring Exporter properties file.
        job_name        - name of the Databricks job.
        job_id          - ID of the Databricks job.
        run_id          - ID of the Databricks run.
        target          - dbt profile target.

    Returns:
        (model_paths, extra_flags, tags)
    """
    dbutils.widgets.text("dbt_project", "")
    dbutils.widgets.text("model_paths", "[]")
    model_paths = json.loads(dbutils.widgets.get("model_paths"))
    if len(model_paths) == 0:
        model_paths = ["silver", "gold"]

    dbutils.widgets.text("extra_flags", "{}")
    extra_flags = json.loads(dbutils.widgets.get("extra_flags"))

    dbutils.widgets.text("tags", "{}")
    tags = json.loads(dbutils.widgets.get("tags"))

    dbutils.widgets.text("properties_file", "")
    dbutils.widgets.text("job_name", "")
    dbutils.widgets.text("job_id", "")
    dbutils.widgets.text("run_id", "")
    dbutils.widgets.text("target", "")

    return model_paths, extra_flags, tags


### Checkpoint Utils

Functions to persist and read the last successfully processed date from a file.
The checkpoint guarantees exactly-once semantics: if a run fails mid-execution
the file is not updated, so the next run retries from the failed date.


In [0]:
def get_last_successful_date(checkpoint_file):
    """Read the last successfully processed date or timestamp from the checkpoint file.

    Returns a datetime for timestamp entries, a date for date-only entries,
    or None if the file does not exist or is empty.
    """
    if not os.path.exists(checkpoint_file):
        return None
    try:
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            first_line = f.readline().strip()
            if first_line:
                for fmt in ("%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
                    try:
                        parsed = datetime.strptime(first_line, fmt)
                        return parsed if " " in first_line else parsed.date()
                    except ValueError:
                        continue
    except Exception as e:
        logging.warning("Error reading checkpoint file: %s", e)
    return None


def generate_date_range(start_date, end_date):
    """Yield every date from start_date to end_date (inclusive)."""
    current = start_date
    while current <= end_date:
        yield current
        current += timedelta(days=1)


def update_checkpoint_file(checkpoint_file, run_date):
    """Persist run_date to the checkpoint file only if more recent than the current value.

    Accepts date objects, datetime objects, or timestamp strings (YYYY-MM-DD HH:MM:SS.ffffff).
    Creates parent directories automatically if they do not exist.
    """
    last_date = get_last_successful_date(checkpoint_file)
    dir_path = os.path.dirname(checkpoint_file)
    if dir_path:
        os.makedirs(dir_path, exist_ok=True)
    if hasattr(run_date, "strftime"):
        date_str = (
            run_date.strftime("%Y-%m-%d %H:%M:%S.%f")
            if isinstance(run_date, datetime)
            else run_date.strftime("%Y-%m-%d")
        )
    else:
        date_str = str(run_date)
    if last_date is None or str(run_date) > str(last_date):
        with open(checkpoint_file, "w", encoding="utf-8") as f:
            f.write(date_str + "\n")
        logging.info("Checkpoint updated to: %s", date_str)
    else:
        logging.info("Checkpoint unchanged: %s (requested %s is not more recent)", last_date, run_date)


### Table Init

`run_table_init` creates a Delta table by parsing the `CREATE TABLE IF NOT EXISTS` statement
from the dbt model pre_hook. Works for both Silver (SCD) and Gold (`materialized='table'`) models.
Requires the model to have a `pre_hook` with the full table DDL.


In [0]:
def run_table_init(table_path=None):
    """Create a table by parsing the CREATE TABLE pre_hook from the dbt model.

    If table_path is None, the path is derived from the manifest node
    (database.schema.name). Pass an explicit table_path when the target
    path is known independently (e.g. Silver, where silver_path comes from a widget).

    Steps:
        1. Run dbt parse to generate manifest.json.
        2. Locate the model node in manifest.json.
        3. Derive table_path from the node if not provided.
        4. Read the model .sql file and extract the CREATE TABLE pre_hook.
        5. Replace {{this}} with table_path and execute via spark.sql().

    Sends RUNNING/SUCCESSFUL/FAILED monitoring logs via send_stage_log.
    """
    import re

    stage = "table_init"
    send_stage_log(stage, LogStatusType.RUNNING, "Initializing table from dbt model pre_hook.")
    try:
        dbt_project_path = dbutils.widgets.get("dbt_project")

        parse_result = dbt_runner.invoke(["parse", "--target", target])
        if not parse_result.success:
            raise RuntimeError(
                f"dbt parse failed: {parse_result.exception or 'unknown error'}"
            )

        manifest_path = f"{dbt_project_path}/target/manifest.json"
        if not os.path.exists(manifest_path):
            raise FileNotFoundError(f"manifest.json not found at '{manifest_path}'")

        with open(manifest_path) as f:
            manifest = json.load(f)

        node = next(
            (
                n
                for mp in model_paths
                for n in manifest["nodes"].values()
                if n["name"] == mp and n["resource_type"] == "model"
            ),
            None,
        )

        if not node:
            raise ValueError(f"No model node found for '{model_paths}' in manifest.json.")

        if table_path is None:
            table_path = f"{node['database']}.{node['schema']}.{node['name']}"

        model_file = f"{dbt_project_path}/{node['original_file_path']}"
        if not os.path.exists(model_file):
            raise FileNotFoundError(f"Model file not found: '{model_file}'")

        with open(model_file) as f:
            model_content = f.read()

        pre_hook_pattern = (
            r"""pre_hook\s*=\s*\[?\s*(["'])\s*"""
            r"""(CREATE\s+TABLE\s+IF\s+NOT\s+EXISTS.*?)"""
            r"""\s*\1"""
        )
        match = re.search(pre_hook_pattern, model_content, re.IGNORECASE | re.DOTALL)
        if not match:
            raise ValueError(
                f"No 'CREATE TABLE IF NOT EXISTS' pre_hook found in '{model_file}'. "
                "Add a pre_hook with the table schema to the dbt model."
            )

        hook_sql = match.group(2)
        create_sql = re.sub(r'\{{\s*this\s*\}\}', table_path, hook_sql)
        if not re.search(r'USING\s+DELTA', create_sql, re.IGNORECASE):
            create_sql += " USING DELTA"

        spark.sql(create_sql)
        logging.info("Table '%s' created.", table_path)
        send_stage_log(stage, LogStatusType.SUCCESSFUL, f"Table '{table_path}' initialized successfully.")

    except Exception as e:
        metrics = HashMap()
        metrics.put("errors", str(e)[:1000])
        send_stage_log(stage, LogStatusType.FAILED, "Failed to initialize table.", metrics)
        raise

### Test Runner

Shared helpers for wrappers that run dbt test with blocking/non-blocking failure classification.
Each test controls its behaviour via meta.blocking in its dbt config (false by default).

| Function | Responsibility |
|---|---|
| categorize_test_results(result) | Splits results into blocking and non-blocking failure lists |
| run_dbt_tests(commands, select, stage) | Invoke, classify, and send monitoring logs for a dbt test run |

In [0]:
def categorize_test_results(result):
    """Split dbt test results into blocking and non-blocking failures.

    A failure is blocking only when the test meta.blocking is explicitly True.
    The default is non-blocking. Warns are always non-blocking.

    Returns:
        (blocking_failures: list[str], non_blocking_failures: list[str])
    """
    blocking_failures = []
    non_blocking_failures = []
    for r in result.result.results:
        if r.status.value not in ("fail", "warn"):
            continue
        try:
            meta = r.node.config.meta or {}
        except AttributeError:
            meta = {}
        is_blocking = meta.get("blocking", False)
        if is_blocking and r.status.value == "fail":
            blocking_failures.append(r.node.unique_id)
        else:
            non_blocking_failures.append(r.node.unique_id)
    return blocking_failures, non_blocking_failures


def run_dbt_tests(commands, select, stage):
    """Invoke dbt test with RUNNING/SUCCESSFUL/FAILED monitoring logs and blocking/non-blocking classification.

    Blocking failures  (meta.blocking=true): raise an exception - job fails.
    Non-blocking failures (meta.blocking=false, default, or warn): log a warning - job continues.
    """
    send_stage_log(stage, LogStatusType.RUNNING, f"Running dbt tests ({select})")
    logging.info("[dbt:%s] Command: dbt %s", stage, " ".join(str(c) for c in commands))
    result = dbt_runner.invoke(commands)
    metrics = HashMap()
    if result.exception:
        metrics.put("errors", str(result.exception)[:1000])
        send_stage_log(stage, LogStatusType.FAILED, "dbt test invocation failed.", metrics)
        raise Exception(f"dbt test failed with exception: {result.exception}")
    blocking_failures, non_blocking_failures = categorize_test_results(result)
    if non_blocking_failures:
        logging.warning("Non-blocking test failures (continuing): %s", non_blocking_failures)
        metrics.put("non_blocking_failures", str(non_blocking_failures))
        success_msg += f" ({len(non_blocking_failures)} non-blocking warning(s))"
    if blocking_failures:
        logging.error("Blocking test failures: %s", blocking_failures)
        metrics = HashMap()
        metrics.put("blocking_failures", str(blocking_failures))
        send_stage_log(stage, LogStatusType.FAILED, f"Tests failed: {len(blocking_failures)} blocking failure(s).", metrics)
        raise Exception(f"Blocking test failures: {blocking_failures}")
    success_msg = "dbt tests passed"
    if non_blocking_failures:
        metrics.put("non_blocking_failures", str(non_blocking_failures))
    send_stage_log(stage, LogStatusType.SUCCESSFUL, success_msg, metrics)

### Initialization

Executed once when the notebook loads. Sets the module-level variables
(`tool`, `process`, `job_id`, `run_id`, `target`, `dbt_runner`,
`scala_tags`, `extra_info`, `model_paths`, `extra_flags`, `tags`)
that all pipeline functions rely on.

In [0]:
model_paths, extra_flags, tags = setup_common_widgets()

timestamp = setup_dbt_env()
tool, process, job_id, run_id, target, dbt_runner = build_common_vars()
scala_tags = build_scala_tags(tags)
extra_info = build_extra_info()
execution_arguments = HashMap()


In [0]:
%scala
import com.brisa.monitoring.MonitoringExporter
import com.brisa.monitoring.model.{Log, LogStatusType}

val propertiesFile = dbutils.widgets.get("properties_file")
MonitoringExporter.initialize(propertiesFile, "DBT", dbutils.widgets.get("job_name"), dbutils.widgets.get("job_id"), dbutils.widgets.get("run_id"), None, None)